In [1]:
import pandas as pd
import os

# 读取数据
input_file = 'TrainTest.csv'

if os.path.exists(input_file):
    print(f"Reading data from: {input_file}")
    df = pd.read_csv(input_file)
    df['time'] = pd.to_datetime(df['time'])
    print(f"Data loaded. Shape: {df.shape}")
else:
    print(f"Error: {input_file} not found.")

Reading data from: TrainTest.csv
Data loaded. Shape: (109622, 5)


In [2]:
# 定义分割逻辑
TEST_POINTS = 72 # 6 hours * 12 points/hour

train_dfs = []
test_dfs = []

# 按受试者分组处理
grouped = df.groupby('id')

print(f"Processing {len(grouped)} subjects...")

for subject_id, group in grouped:
    # 确保按时间排序
    group = group.sort_values('time')
    
    n_samples = len(group)
    
    if n_samples <= TEST_POINTS:
        print(f"Warning: Subject {subject_id} has only {n_samples} points (<= {TEST_POINTS}). All assigned to Test.")
        test_dfs.append(group)
    else:
        # 切分
        train_data = group.iloc[:-TEST_POINTS]
        test_data = group.iloc[-TEST_POINTS:]
        
        train_dfs.append(train_data)
        test_dfs.append(test_data)

# 合并
df_train = pd.concat(train_dfs, ignore_index=True)
df_test = pd.concat(test_dfs, ignore_index=True)

print("Split complete.")
print(f"Train set shape: {df_train.shape}")
print(f"Test set shape:  {df_test.shape}")

Processing 164 subjects...
Split complete.
Train set shape: (97814, 5)
Test set shape:  (11808, 5)


In [3]:
# 导出数据
output_train = 'Train.csv'
output_test = 'Test.csv'

df_train.to_csv(output_train, index=False)
df_test.to_csv(output_test, index=False)

print(f"Saved Train set to: {os.path.abspath(output_train)}")
print(f"Saved Test set to:  {os.path.abspath(output_test)}")

Saved Train set to: c:\Users\江一骏\Desktop\学习\HKU\dissertation\engineering\code\GlucosePrediction\src\DataSplit\TrainTest\Train.csv
Saved Test set to:  c:\Users\江一骏\Desktop\学习\HKU\dissertation\engineering\code\GlucosePrediction\src\DataSplit\TrainTest\Test.csv


## 总结

- **输入**: `TrainTest.csv`
- **输出**:
    - `Train.csv`: 用于模型训练。
    - `Test.csv`: 用于模型评估 (每个受试者的最后 6 小时)。